In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/d2c_marketing_funnel_data.csv")

# The first 5 rows to understand the column names and data format
print(df.head())

# Check column names, non-null counts, and data types
print(df.info())

   user_id  session_id        date    month   channel campaign_type  device  \
0   221958           1   8/16/2025  2025-08   Organic    New Launch  Mobile   
1   771155           2  12/16/2025  2025-12   Organic    Influencer  Mobile   
2   231932           3   7/17/2025  2025-07   Organic    Influencer  Mobile   
3   465838           4    7/4/2025  2025-07  Paid Ads      Discount  Mobile   
4   359178           5   8/10/2025  2025-08  Paid Ads    Influencer  Mobile   

   user_type     region visited_website viewed_product added_to_cart  \
0        New      Metro             Yes             No            No   
1        New  Non-Metro             Yes            Yes           Yes   
2        New  Non-Metro             Yes            Yes            No   
3  Returning      Metro             Yes            Yes           Yes   
4  Returning  Non-Metro             Yes             No            No   

  checkout_started purchase_completed discount_applied  order_value   revenue  
0           

In [ ]:
# Define the sequential funnel stages in order
stages = [
    "visited_website",
    "viewed_product",
    "added_to_cart",
    "checkout_started",
    "purchase_completed",
]

# Calculate the total count of 'Yes' for each stage
funnel_counts = {}
for stage in stages:
  # Count how many rows have 'Yes' for this specific stage column
  count = (df[stage] == "Yes").sum()
  funnel_counts[stage] = count

# Convert to a clean Pandas DataFrame for display
funnel_df = pd.DataFrame(
    list(funnel_counts.items()), columns=["Funnel_Stage", "User_Count"]
)

# Calculate the conversion rate from the initial visit (Stage 1)
total_visits = funnel_df.loc[
    funnel_df["Funnel_Stage"] == "visited_website", "User_Count"
].values[0]
funnel_df["Conversion_From_Start_%"] = (
    funnel_df["User_Count"] / total_visits
) * 100

# Calculate stage-to-stage drop-off/conversion rate
funnel_df["Stage_To_Stage_%"] = (
    funnel_df["User_Count"] / funnel_df["User_Count"].shift(1)
) * 100
funnel_df.loc[0, "Stage_To_Stage_%"] = (
    100.0  # First stage is always 100% of visits
)

# Round numbers for clean viewing
funnel_df["Conversion_From_Start_%"] = funnel_df[
    "Conversion_From_Start_%"
].round(2)
funnel_df["Stage_To_Stage_%"] = funnel_df["Stage_To_Stage_%"].round(2)

print(funnel_df)

         Funnel_Stage  User_Count  Conversion_From_Start_%  Stage_To_Stage_%
0     visited_website      120000                   100.00            100.00
1      viewed_product       77870                    64.89             64.89
2       added_to_cart       27156                    22.63             34.87
3    checkout_started       16234                    13.53             59.78
4  purchase_completed        8181                     6.82             50.39


In [21]:
import plotly.express as px

# Create a clean label column showing only the conversion percentage from the start
funnel_df["Percent_Label"] = funnel_df["Conversion_From_Start_%"].apply(
    lambda x: f"{x}%"
)

# Build the interactive funnel chart using User_Count for the bar width
fig = px.funnel(
    funnel_df,
    x="User_Count",
    y="Funnel_Stage",
    title="Direct-to-Consumer E-Commerce Conversion Funnel",
    text="Percent_Label",
    labels={
        "User_Count": "Number of Users",
        "Funnel_Stage": "Stage of Funnel",
    },
)

# Format text size and position to keep it large and clean
fig.update_traces(
    textposition="inside", textinfo="text", textfont=dict(size=14)
)

# Clean up layout and remove the Y-axis title from the left side
fig.update_layout(
    title_x=0.5,
    height=600,
    width=900,
    yaxis_title=""  # This removes "Stage of Funnel" from the side
)

# Display the interactive chart
fig.show()

In [ ]:
# Group by device and calculate total visits and purchases
device_funnel = (
    df.groupby("device")
    .agg(
        total_visits=("visited_website", lambda x: (x == "Yes").count()),
        total_purchases=(
            "purchase_completed",
            lambda x: (x == "Yes").sum(),
        ),
    )
    .reset_index()
)

# Calculate conversion rate per device
device_funnel["conversion_rate_%"] = (
    device_funnel["total_purchases"] / device_funnel["total_visits"]
) * 100
device_funnel["conversion_rate_%"] = device_funnel[
    "conversion_rate_%"
].round(2)

print(device_funnel)

    device  total_visits  total_purchases  conversion_rate_%
0  Desktop         35994             2488               6.91
1   Mobile         84006             5693               6.78
